In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

from tqdm import tqdm

from pyhnc import *

Set up basic variables for Ornstein-Zernike solver.

In [ ]:
N = 2**18
L = 2000
grid = Grid(L, N)
r, q = grid.r, grid.q

verbose = True

alpha = 0.5
niters = 1000
tol = 1e-12
solvent = Solver(grid, alpha=alpha, niters=niters, tol=tol)

A0, rcut = 25, 1
v0 = potentials.DPD(A0, rcut)

In [ ]:
A0, rcut = 25, 1
v0 = potentials.DPD(A0, rcut)
ρ0 = 3.0

sol0 = solvent.solve(v0, ρ0, monitor=verbose)

In [ ]:
verbose = False
xmin = 1e-4
xmax = 0.2
N = 51
xz = np.geomspace(xmin, xmax, N)
xw = 1 - xz
mu_ref = sol0.excess_chemical_potential
m = 3

A0 = 25
rcut = 1
sigma = 0.5
alpha = 1/(2*sigma**2)

Aij = A0 * np.ones((m, m))
v1 = potentials.DPDGaussianIon(Aij, [0, 1, -1], alpha, rcut)
print(Aij)

rho1 = np.empty(xz.size)
p1 = np.empty(xz.size)

first = True
eps = 1e-3
ρ = sol0.density

for i, x in enumerate(tqdm(xz)):

    def solve(rho):
        ρz = x * rho
        ρ0 = rho - ρz
        ρi = np.array([ρ0, 0.5*ρz, 0.5*ρz])
        assert np.isclose(np.sum(ρi), rho)
        return solvent.solve(v1, ρi, monitor=verbose, restart=first)

    def residual(rho):
        sol = solve(rho)
        mu0 = sol.excess_chemical_potential[0]
        return (mu0 + np.log((1-x)*rho/sol0.density) - mu_ref)**2

    ρ = minimize_scalar(residual, bounds=(ρ-eps, ρ+eps), method='bounded').x
    rho1[i] = ρ
    p1[i] = solve(ρ).pressure
    first = False


In [ ]:
verbose = False
rho2 = np.empty(xz.size)
p2 = np.empty(xz.size)

Aij = A0 * np.ones((m, m))
Aij[0,1:] += 5
Aij[1:,0] = Aij[0,1:]
v2 = potentials.DPDGaussianIon(Aij, [0, 1, -1], alpha, rcut)
print(Aij)

first = True
eps = 1e-3
ρ = sol0.density

for i, x in enumerate(tqdm(xz)):

    def solve(rho):
        ρz = x * rho
        ρ0 = rho - ρz
        ρi = np.array([ρ0, 0.5*ρz, 0.5*ρz])
        assert np.isclose(np.sum(ρi), rho)
        return solvent.solve(v2, ρi, monitor=verbose, restart=first)

    def residual(rho):
        sol = solve(rho)
        mu0 = sol.excess_chemical_potential[0]
        return (mu0 + np.log((1-x)*rho/sol0.density) - mu_ref)**2

    ρ = minimize_scalar(residual, bounds=(ρ-eps, ρ+eps), method='bounded').x
    rho2[i] = ρ
    p2[i] = solve(ρ).pressure
    first = False


In [ ]:
c1 = xz * rho1
phi1 = (p1 - sol0.pressure)/c1
pl1, = plt.plot(xz, phi1)

c2 = xz * rho2
phi2 = (p2 - sol0.pressure)/c2
pl2, = plt.plot(xz, phi2, '--')

def matrix(A):
    s = r'\\'.join(['&'.join([f'{aij:.0f}' for aij in row]) for row in A])
    return rf'$A_{{ij}} = \begin{{pmatrix}}{s}\end{{pmatrix}}$'

plt.text(0.025, 0.15, matrix(v1.dpd.A), c=pl1.get_color(),
         ha='left', va='center', fontsize=8,
         transform=plt.gca().transAxes)
plt.text(1, 0.85, matrix(v2.dpd.A), c=pl2.get_color(),
         ha='right', va='center', fontsize=8,
         transform=plt.gca().transAxes)

plt.axhline(y=1, ls=':')
plt.axvline(x=0, ls=':')

plt.ylim([0, 2])
plt.xlim([0, xmax])
# plt.gca().set_xscale('log')
# plt.gca().set_yscale('log')

plt.legend(loc='best')
plt.xlabel(r'mole fraction $x_z \equiv x_+ + x_-$')
plt.ylabel(r'osmotic coefficient $\Phi = \beta \Pi / c_z$')
plt.show()

In [ ]:
pl1, = plt.plot(xz, 1-phi1)
pl2, = plt.plot(xz, 1-phi2, '--')

plt.axhline(y=1, ls=':')
plt.axvline(x=0, ls=':')

# plt.xlim([0, xmax])
plt.gca().set_xscale('log')
plt.gca().set_yscale('log')
plt.xlim([1e-4, 0.2])
# plt.ylim([2e-3, 5e-1])


k = np.sqrt(4*np.pi * v1.ion.lB * xz * rho1)
plt.plot(xz, v1.ion.lB * k / 6, ':', c=pl1.get_color(),
         label=r'Debye-H\"uckel limiting law')


from mpltools import annotation
f = lambda x,c: c*x**0.5
x, c = 5e-4, 1.1
annotation.slope_marker((x, f(x, c)), (1, 2), invert=True, ax=plt.gca(),
                        poly_kwargs=dict(ec='black', fill=False, lw=0.5),
                        text_kwargs=dict(fontsize=8))


plt.legend(loc='best')
plt.xlabel(r'mole fraction $x_z \equiv x_+ + x_-$')
plt.ylabel(r'$1-\Phi$')
plt.show()

In [ ]:
# pl1, = plt.plot(xz, 1-phi1, label=r'$N=2^{16}, L=1000$')
# pl2, = plt.plot(xz, 1-phi2, '--', label=r'$N=2^{18}, L=2000$')

# plt.axhline(y=1, ls=':')
# plt.axvline(x=0, ls=':')

# # plt.xlim([0, xmax])
# plt.gca().set_xscale('log')
# plt.gca().set_yscale('log')
# plt.xlim([1e-5, 1e-1])
# plt.ylim([2e-3, 1e-1])


# k = np.sqrt(4*np.pi * v1.ion.lB * xz * rho1)
# plt.plot(xz, v1.ion.lB * k / 6, ':', c=pl1.get_color(),
#          label=r'Debye-H\"uckel limiting law')


# from mpltools import annotation
# f = lambda x,c: c*x**0.5
# x, c = 5e-5, 1.1
# annotation.slope_marker((x, f(x, c)), (1, 2), invert=True, ax=plt.gca(),
#                         poly_kwargs=dict(ec='black', fill=False, lw=0.5),
#                         text_kwargs=dict(fontsize=8))


# plt.legend(loc='best')
# plt.xlabel(r'mole fraction $x_z \equiv x_+ + x_-$')
# plt.ylabel(r'$1-\Phi$')
# plt.show()